# 第一讲：NumPy 数组的本质

**学习目标**
- 理解为什么 Python 列表对数值计算来说太慢了
- 掌握 ndarray 的内存布局与 dtype 体系
- 熟练使用各种数组创建函数

---

## 1.1 问题的起点——Python 列表为什么慢？

我们先看一个具体现象：

In [ ]:
import numpy as np
import time
import sys

# 创建 100 万个 int 的列表 vs 数组
n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

# 对比 1：内存占用
list_size = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list[:1000]) * (n // 1000)
arr_size = np_arr.nbytes

print(f"Python list 内存 (估): {list_size / 1024 / 1024:.1f} MB")
print(f"NumPy array 内存:      {arr_size / 1024 / 1024:.1f} MB")
print()

# 对比 2：运算速度
t0 = time.perf_counter()
s1 = sum(x * 2 for x in py_list)
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
s2 = np.sum(np_arr * 2)
t_arr = time.perf_counter() - t0

print(f"Python list 求和 ×2: {t_list*1000:.1f} ms")
print(f"NumPy array 求和 ×2: {t_arr*1000:.1f} ms")
print(f"加速: {t_list / t_arr:.0f}×")

两个问题浮出水面：

1. **内存膨胀**：Python 中每个整数都是一个完整的 Python 对象（28 字节），而 NumPy 中只是一个 8 字节的 C `int64`
2. **速度差距**：Python 循环每次都要经过解释器、类型检查、方法分派，NumPy 的循环直接跑在编译好的 C 代码里

### 图解：两种内存布局

```
Python list [1, 2, 3, 4]:
┌───────┬───────┬───────┬───────┐
│  ptr  │  ptr  │  ptr  │  ptr  │  ← 指针数组（连续）
└───┼───┴───┼───┴───┼───┴───┼───┘
    │       │       │       │
    ▼       ▼       ▼       ▼
  ┌─────┐┌─────┐┌─────┐┌─────┐
  │ 1   ││ 2   ││ 3   ││ 4   │  ← Python int 对象（散落在堆上）
  │ref=1││ref=1││ref=1││ref=1│    每个 28 字节
  │type ││type ││type ││type │
  └─────┘└─────┘└─────┘└─────┘

NumPy array([1, 2, 3, 4], dtype=int64):
┌───┬───┬───┬───┐
│ 1 │ 2 │ 3 │ 4 │  ← 纯数据（连续 32 字节 = 4 × 8 bytes）
└───┴───┴───┴───┘    没有对象头、引用计数、类型指针
```

这就是 NumPy 一切性能优势的**物理根源**。后面讲的所有技巧——ufunc、广播、向量化——都建立在「数据连续存储」这一前提之上。

## 1.2 ndarray 的三要素

每个 NumPy 数组由三个核心属性定义：

In [ ]:
arr = np.array([[1, 2, 3],
                [4, 5, 6]], dtype=np.float64)

print("ndarray 的三要素:")
print("  shape  (形状):", arr.shape)      # 各维度的尺寸
print("  dtype  (数据类型):", arr.dtype)   # 每个元素的类型
print("  strides (步长):", arr.strides)    # 每维移动一个「位置」需要跨过的字节数
print()
print("其他属性:")
print("  ndim   (维度数):", arr.ndim)
print("  size   (总元素数):", arr.size)
print("  nbytes (总字节数):", arr.nbytes)
print("  data   (内存地址):", arr.ctypes.data)

**strides 是理解 NumPy 进阶操作的关键。** 它解释了为什么转置、切片、广播能够「零拷贝」——这些操作只是修改了 shape 和 strides，底层数据根本没动。

```
arr = [[1, 2, 3],
       [4, 5, 6]]           dtype=float64, shape=(2, 3)

内存中的实际布局（行优先，C order）：
  [1] [2] [3] [4] [5] [6]   ← 连续 48 字节

strides = (24, 8)：
  · 从一行到下一行：跨 24 字节（3 个元素）
  · 从一列到下一列：跨 8 字节 （1 个元素）
```

In [ ]:
# 用 strides 理解转置的「零拷贝」
arr = np.array([[1, 2, 3],
                [4, 5, 6]], dtype=np.float64)

print("原始数组:")
print(f"  shape: {arr.shape}, strides: {arr.strides}")

arr_t = arr.T  # 转置
print("\n转置后:")
print(f"  shape: {arr_t.shape}, strides: {arr_t.strides}")
print()
print("注意：strides 从 (24, 8) 变成了 (8, 24)")
print("数据本身没有被移动——只是行列的步长交换了！")

# 验证：修改原始数组，转置视图也会变
arr[0, 0] = 999
print("\n修改 arr[0,0]=999 后，arr_t 也变了:")
print(arr_t)

## 1.3 dtype 体系——给每个比特赋予含义

dtype 决定了：
- 一个元素占几个字节
- 这些字节被解释成整数还是浮点数
- 能表示的范围和精度

选择正确的 dtype 不是可选的——它直接影响计算的正确性、性能和内存。

In [ ]:
# === 整数类型族 ===
print("=== 整数类型 ===")
for dtype in [np.int8, np.int16, np.int32, np.int64,
              np.uint8, np.uint16, np.uint32, np.uint64]:
    info = np.iinfo(dtype)
    print(f"  {dtype.__name__:8s}: {info.bits:2d} bits, "
          f"范围 [{info.min}, {info.max}]")

In [ ]:
# === 浮点类型族 ===
print("=== 浮点类型 ===")
for dtype in [np.float16, np.float32, np.float64]:
    info = np.finfo(dtype)
    print(f"  {dtype.__name__:8s}: {info.bits:2d} bits, "
          f"精度≈{info.precision+1} 位有效数字, "
          f"最大值≈{info.max:.2e}")

In [ ]:
# === 一个经典陷阱：整数溢出 ===
a = np.array([200], dtype=np.int8)   # int8 最大只能到 127
print("int8 200 的实际存储值:", a)   # 溢出回绕，变成 -56 或类似值

b = np.array([200], dtype=np.int64)
print("int64 200 的实际存储值:", b)

# === 另一个经典陷阱：dtype 自动推断 ===
print()
arr = np.array([1, 2, 3])          # 全是整数 → 自动 int64
print(f"[1,2,3] → dtype={arr.dtype}")

arr = np.array([1, 2, 3.0])        # 混入浮点数 → 自动 float64
print(f"[1,2,3.0] → dtype={arr.dtype}")

arr = np.array([1, 2, '3'])        # 混入字符串 → 自动转为 <U21 (Unicode 字符串)
print(f"[1,2,'3'] → dtype={arr.dtype}")

print("\n教训：创建数组时显式指定 dtype！")

### dtype 选择指南

| 场景 | 推荐 dtype | 原因 |
|------|-----------|------|
| 通用科学计算 | `float64` | 默认、精度够、性能好 |
| 深度学习推理 | `float16` / `bfloat16` | 省一半内存，GPU 加速 |
| 大规模计数/索引 | `int32` / `uint32` | 范围够用（21 亿/43 亿），省一半内存 |
| 图像像素（0-255） | `uint8` | 每个像素仅 1 字节 |
| 布尔掩码 | `bool` | 每元素 1 字节（比 int 省 8 倍） |
| 金融计算 | `float64` 或 `decimal` | 避免 float32 的精度损失 |

## 1.4 数组创建——10 个最常用的函数

掌握这些函数是 NumPy 基本功。我们按「从无到有」和「从现有数据」两类来讲。

In [ ]:
# ========== 第一类：从无到有创建 ==========

# 1. np.array() — 从 Python 序列创建（显式指定 dtype！）
arr = np.array([1, 2, 3, 4, 5], dtype=np.float64)
print("1. np.array:", arr)

# 2. np.arange(start, stop, step) — 等步长序列
print("2. np.arange(0, 10, 2):", np.arange(0, 10, 2))

# 3. np.linspace(start, stop, num) — 等分序列（包含终点）
print("3. np.linspace(0, 1, 5):", np.linspace(0, 1, 5))

# 4. np.zeros(shape) / np.ones(shape) — 全零/全一
print("4. np.zeros((2, 3)):\n", np.zeros((2, 3)))
print("   np.ones((2, 3)):\n", np.ones((2, 3)))

# 5. np.full(shape, value) — 全填充指定值
print("5. np.full((2, 3), 7.5):\n", np.full((2, 3), 7.5))

# 6. np.eye(N) / np.identity(N) — 单位矩阵
print("6. np.eye(3):\n", np.eye(3))

# 7. np.empty(shape) — 只分配内存不初始化（垃圾数据）
print("7. np.empty(3):", np.empty(3), "← 注意这是内存中的随机值")

In [ ]:
# ========== 第二类：从已有数据创建 ==========

# 8. np.copy(arr) — 深拷贝
orig = np.array([1, 2, 3])
cpy = np.copy(orig)
cpy[0] = 999
print(f"8. 深拷贝: orig={orig}, cpy={cpy} (互不影响)")

# 9. astype() — 转换 dtype（同时拷贝数据）
a = np.array([1.7, 2.3, 3.9])
print(f"9. float → int (截断): {a.astype(np.int64)}")

# 10. reshape() — 改变形状（不拷贝数据）
a = np.arange(12)
b = a.reshape(3, 4)   # 总元素数必须一致
print(f"10. reshape (1, 12) → (3, 4):\n{b}")

# reshape 的自动推断：用 -1 让 NumPy 帮你算
b = a.reshape(3, -1)   # 3 行，列数自动算
print(f"    reshape (3, -1): shape={b.shape}")
b = a.reshape(-1, 2)   # 2 列，行数自动算
print(f"    reshape (-1, 2): shape={b.shape}")

In [ ]:
# ========== 第三类：特殊用途 ==========

# np.meshgrid — 生成网格坐标
x = np.linspace(-1, 1, 3)
y = np.linspace(-2, 2, 3)
X, Y = np.meshgrid(x, y)
print("meshgrid 结果:")
print(f"X (3×3):\n{X}")
print(f"Y (3×3):\n{Y}")
print("\n这样就可以一次性计算 Z = f(X, Y) 了！")
Z = X**2 + Y**2  # 逐元素运算，不需要任何循环
print(f"Z = X²+Y²:\n{Z}")

In [ ]:
# np.concatenate / np.stack — 拼接数组
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("a:\n", a, "\nb:\n", b)
print()
print("np.concatenate([a,b], axis=0) [纵向堆叠]:")
print(np.concatenate([a, b], axis=0))
print()
print("np.concatenate([a,b], axis=1) [横向堆叠]:")
print(np.concatenate([a, b], axis=1))
print()
print("np.stack([a,b], axis=0) [新增维度堆叠]:")
stacked = np.stack([a, b], axis=0)
print(f"  shape: {stacked.shape}")
print(stacked)

## 1.5 视图（View）与拷贝（Copy）——最容易出错的陷阱

**基本法则**：切片返回**视图**，花式索引返回**拷贝**。搞混这两者会让你 debug 到怀疑人生。

In [ ]:
arr = np.arange(10)
print("原始 arr:", arr)

# 切片 → 视图
s = arr[2:5]
s[0] = 999
print("切片 s=arr[2:5] 并修改 s[0]=999:")
print("  arr:", arr, "← 原数组也被改了！")

# 花式索引 → 拷贝
arr = np.arange(10)  # 重置
f = arr[[2, 3, 4]]
f[0] = 999
print("\n花式索引 f=arr[[2,3,4]] 并修改 f[0]=999:")
print("  arr:", arr, "← 原数组没变")

# 如何判断是视图还是拷贝？
print("\n判断 view/copy:")
print("  切片 arr[2:5].base is arr:", s.base is arr)     # True → 视图
print("  花式索引 f.base is arr:", f.base is arr if f.base is not None else "None → 拷贝")

## 1.6 C-order vs Fortran-order

NumPy 默认以 C 风格（行优先）存储数据。这意味着**最后一维变化最快**。

In [ ]:
arr = np.array([[1, 2, 3],
                [4, 5, 6]])

print("形状:", arr.shape)
print("内存中实际排列（C-order，行优先）:")
print("  ", arr.ravel())  # 展平后看到实际存储顺序
print("  即: 1, 2, 3 (第0行) → 4, 5, 6 (第1行)")

print("\nFortran-order（列优先）:")
arr_f = np.asfortranarray(arr)
print("  ", arr_f.ravel(order='K'))
print("  即: 1, 4 (第0列) → 2, 5 (第1列) → 3, 6 (第2列)")

**这对性能有一阶影响。** 沿行遍历（连续内存）比沿列遍历（跳跃内存）快数倍：

In [ ]:
n = 10000
mat = np.random.randn(n, n)

# 沿行求和（内存连续）
t0 = time.perf_counter()
_ = mat.sum(axis=1)
t_row = time.perf_counter() - t0

# 沿列求和（内存跳跃）
t0 = time.perf_counter()
_ = mat.sum(axis=0)
t_col = time.perf_counter() - t0

print(f"沿行求和 (axis=1, 连续内存): {t_row*1000:.1f} ms")
print(f"沿列求和 (axis=0, 跳跃内存): {t_col*1000:.1f} ms")
print(f"差异: {t_col / t_row:.1f}×")

## 本讲小结

1. NumPy 快的根源是**连续内存 + C 循环**，不是什么魔法
2. `shape`, `dtype`, `strides` 是 ndarray 的灵魂三要素
3. 切片是**视图**（共享数据），花式索引是**拷贝**
4. 始终显式指定 dtype，别让 NumPy 替你猜
5. 内存连续性对性能有一阶影响——能沿行操作就别沿列

---

**思考题**

1. 为什么 `np.zeros((3, 4)).strides` 是 `(32, 8)` 而不是 `(32, 8)`？（提示：默认 dtype 是什么？）
2. 切片返回视图，那 `arr[0:5][0:3]` 修改后会影响原数组吗？为什么？